In [1]:


import pandas as pd
import numpy as np
import joblib
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings("ignore")

# ── Recharger les données préparées ─────────────────────
df_train = pd.read_csv("../data/train.csv")
df_val   = pd.read_csv("../data/val.csv")
df_test  = pd.read_csv("../data/test.csv")
tfidf    = joblib.load("../models/tfidf.pkl")

def clean_text(text):
    import re
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

KEYWORDS = ["ignore","bypass","jailbreak","forget","pretend",
            "roleplay","override","disregard","base64","system"]

def manual_features(df):
    feats = pd.DataFrame()
    feats["text_len"]        = df["text"].str.len()
    feats["word_count"]      = df["text"].str.split().str.len()
    feats["has_keyword"]     = df["text"].str.lower().apply(lambda x: int(any(k in str(x) for k in KEYWORDS)))
    feats["has_base64"]      = df["text"].str.contains(r"[A-Za-z0-9+/]{20,}={0,2}", regex=True).astype(int)
    feats["uppercase_ratio"] = df["text"].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)),1))
    return feats

for df in [df_train, df_val, df_test]:
    df["clean_text"] = df["text"].apply(clean_text)

X_train = hstack([tfidf.transform(df_train["clean_text"]), csr_matrix(manual_features(df_train).values)])
X_val   = hstack([tfidf.transform(df_val["clean_text"]),   csr_matrix(manual_features(df_val).values)])
X_test  = hstack([tfidf.transform(df_test["clean_text"]),  csr_matrix(manual_features(df_test).values)])

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values

In [4]:
# ── Définir les 4 modèles ────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "LinearSVC":           LinearSVC(max_iter=2000, C=1.0),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    print(f"\n Entraînement : {name} ...")
    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred_val)
    f1  = f1_score(y_val, y_pred_val)

    results[name] = {"accuracy": acc, "f1": f1, "model": model}
    print(f"   Val Accuracy: {acc:.4f} | F1: {f1:.4f}")

    # Sauvegarder
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(model, f"../models/{safe_name}.pkl")
    print(f"  Sauvegardé : models/{safe_name}.pkl")


 Entraînement : Logistic Regression ...
   Val Accuracy: 0.9469 | F1: 0.9546
  Sauvegardé : models/logistic_regression.pkl

 Entraînement : Random Forest ...
   Val Accuracy: 0.9522 | F1: 0.9593
  Sauvegardé : models/random_forest.pkl

 Entraînement : LinearSVC ...
   Val Accuracy: 0.9501 | F1: 0.9570
  Sauvegardé : models/linearsvc.pkl

 Entraînement : Gradient Boosting ...
   Val Accuracy: 0.9384 | F1: 0.9464
  Sauvegardé : models/gradient_boosting.pkl
